# 05 — MLOps: Drift Detection & Monitoring

**Objective:** Simulate production drift, run weekly PSI + KS checks, visualise results, and validate the automatic retraining trigger pipeline.

**Drift types simulated:**
- Week 1: Normal production (should be stable)
- Week 2: Moderate drift (slight distribution shift)
- Week 3: Severe drift (large shift → retraining triggered)

**Sections:**
1. Load baseline and simulate production data
2. PSI analysis per sensor
3. KS test analysis
4. Weekly drift pipeline (orchestrator)
5. Drift visualisation
6. Alert logs review

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.dataset import load_splits
from src.monitoring.psi_monitor import PSIMonitor, calculate_psi
from src.monitoring.ks_monitor import KSMonitor
from src.monitoring.drift_pipeline import DriftPipeline
from src.monitoring.alerts import send_alert

sns.set_style("darkgrid")
os.makedirs("../logs/drift_reports", exist_ok=True)
print("Imports OK")

## 1. Load Baseline & Simulate Three Production Weeks

In [ ]:
splits = load_splits("../data/processed")
X_train = splits["X_train"]   # baseline — normal sequences used for training

# ── Simulate 3 production weeks ─────────────────────────────────────────────
rng = np.random.default_rng(99)

# Week 1: Identical to training distribution (no drift)
idx_w1 = rng.choice(len(X_train), size=500, replace=False)
X_week1 = X_train[idx_w1].copy()
np.save("../data/production/X_week_1.npy", X_week1)

# Week 2: Moderate drift (5 sensors shifted by 1σ)
X_week2 = X_train[rng.choice(len(X_train), size=500, replace=False)].copy()
X_week2[:, :, 0:5] += 1.0  # moderate shift on sensors 0-4
np.save("../data/production/X_week_2.npy", X_week2)

# Week 3: Severe drift (15 sensors shifted by 3σ + scale change)
X_week3 = X_train[rng.choice(len(X_train), size=500, replace=False)].copy()
X_week3[:, :, 0:15] += 3.0   # large mean shift
X_week3[:, :, 5:10] *= 2.0   # scale change
np.save("../data/production/X_week_3.npy", X_week3)

os.makedirs("../data/production", exist_ok=True)
print("Production simulation:")
print(f"  Week 1 (stable)     : {X_week1.shape}")
print(f"  Week 2 (mod. drift) : {X_week2.shape}  (sensors 0-4 shifted +1σ)")
print(f"  Week 3 (sev. drift) : {X_week3.shape}  (sensors 0-15 shifted +3σ, sensors 5-10 ×2)")

## 2. PSI Analysis

**PSI thresholds:**
- PSI < 0.10  → OK (no drift)
- PSI 0.10-0.25 → WARNING (monitor)
- PSI ≥ 0.25  → CRITICAL (retrain)

In [ ]:
psi_monitor = PSIMonitor(X_train, psi_threshold=0.25)

psi_w1, crit_w1 = psi_monitor.check_all_sensors(X_week1)
print(f"\nWeek 1 — Critical: {len(crit_w1)} sensors")

psi_w2, crit_w2 = psi_monitor.check_all_sensors(X_week2)
print(f"Week 2 — Critical: {len(crit_w2)} sensors")

psi_w3, crit_w3 = psi_monitor.check_all_sensors(X_week3)
print(f"Week 3 — Critical: {len(crit_w3)} sensors")

In [ ]:
# Extract PSI values for all sensors across 3 weeks
n_sensors = 50

def extract_psi_values(psi_results):
    return [psi_results[f"sensor_{i}"]["psi"] for i in range(n_sensors)]

psi_vals_w1 = extract_psi_values(psi_w1)
psi_vals_w2 = extract_psi_values(psi_w2)
psi_vals_w3 = extract_psi_values(psi_w3)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, vals, title, week in zip(
    axes,
    [psi_vals_w1, psi_vals_w2, psi_vals_w3],
    ["Week 1 (Stable)", "Week 2 (Moderate Drift)", "Week 3 (Severe Drift)"],
    [1, 2, 3]
):
    colors = ["#F44336" if v >= 0.25 else "#FF9800" if v >= 0.10 else "#4CAF50" for v in vals]
    ax.bar(range(n_sensors), vals, color=colors, alpha=0.85)
    ax.axhline(0.25, color="red",    linestyle="--", lw=1.5, label="Critical (0.25)")
    ax.axhline(0.10, color="orange", linestyle=":",  lw=1.5, label="Warning (0.10)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Sensor index")
    ax.set_ylabel("PSI" if ax == axes[0] else "")
    ax.legend(fontsize=8)

plt.suptitle("Population Stability Index per Sensor — 3 Production Weeks",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../logs/drift_reports/psi_weekly_report.png", dpi=120, bbox_inches="tight")
plt.show()

## 3. KS Test Analysis

In [ ]:
ks_monitor = KSMonitor(X_train, alpha=0.05)

ks_w1 = ks_monitor.check_all_sensors(X_week1)
ks_w2 = ks_monitor.check_all_sensors(X_week2)
ks_w3 = ks_monitor.check_all_sensors(X_week3)

print(f"KS drifted sensors:")
print(f"  Week 1: {len(ks_w1):2d} sensors")
print(f"  Week 2: {len(ks_w2):2d} sensors")
print(f"  Week 3: {len(ks_w3):2d} sensors")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

weeks = ["Week 1\n(Stable)", "Week 2\n(Moderate)", "Week 3\n(Severe)"]
ks_counts   = [len(ks_w1), len(ks_w2), len(ks_w3)]
psi_critcounts = [len(crit_w1), len(crit_w2), len(crit_w3)]

x = np.arange(3)
width = 0.35
ax.bar(x - width/2, psi_critcounts, width, label="PSI Critical sensors",
       color="#F44336", alpha=0.85)
ax.bar(x + width/2, ks_counts,      width, label="KS Drifted sensors",
       color="#FF9800", alpha=0.85)
ax.axhline(5,  color="red",    linestyle="--", lw=1.5, label="Retrain PSI threshold (5)")
ax.axhline(10, color="orange", linestyle=":",  lw=1.5, label="Retrain KS threshold (10)")

ax.set_xticks(x)
ax.set_xticklabels(weeks)
ax.set_ylabel("Number of drifted sensors")
ax.set_title("Drift Detection Summary: PSI vs KS Test (3 Weeks)",
             fontsize=13, fontweight="bold")
ax.legend()

for i, (psi_c, ks_c) in enumerate(zip(psi_critcounts, ks_counts)):
    ax.text(i - width/2, psi_c + 0.3, str(psi_c), ha="center", fontweight="bold")
    ax.text(i + width/2, ks_c + 0.3,  str(ks_c),  ha="center", fontweight="bold")

plt.tight_layout()
plt.savefig("../logs/drift_reports/drift_summary.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Weekly Drift Pipeline Orchestration

The `DriftPipeline` triggers retraining when:
- ≥ 5 sensors have PSI ≥ 0.25 (severe data drift), **OR**
- ≥ 10 sensors have KS p-value < 0.05 (widespread distributional shift)

In [ ]:
pipeline = DriftPipeline(
    X_baseline=X_train,
    psi_threshold=0.25,
    ks_alpha=0.05,
    report_dir="../logs/drift_reports",
)

print("\n" + "═"*65)
print("  Running weekly drift checks for all 3 simulated weeks…")
print("═"*65)

reports = {}
for week_num, X_prod in [(1, X_week1), (2, X_week2), (3, X_week3)]:
    label = f"2026-W{week_num:02d}"
    report = pipeline.weekly_check(X_prod, week_label=label)
    reports[week_num] = report

print("\n" + "═"*65)
print("  DRIFT PIPELINE SUMMARY")
print("═"*65)
for wk, rep in reports.items():
    emoji = "🔄" if rep["retrain"] else "✅"
    print(f"  {emoji} Week {wk}  PSI_critical={rep['psi_critical']:2d}  "
          f"KS_drifted={rep['ks_drifted']:2d}  retrain={rep['retrain']}")

## 5. Distribution Comparison — Week 1 vs Week 3

In [ ]:
# Compare sensor 0 distribution across baseline and production weeks
X_base_flat = X_train.reshape(-1, 50)
X_w1_flat   = X_week1.reshape(-1, 50)
X_w3_flat   = X_week3.reshape(-1, 50)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for sidx, ax in zip([0, 5, 12], axes):
    ax.hist(X_base_flat[:, sidx], bins=50, alpha=0.6, label="Baseline",
            color="#2196F3", density=True)
    ax.hist(X_w1_flat[:, sidx],   bins=50, alpha=0.6, label="Week 1 (stable)",
            color="#4CAF50", density=True)
    ax.hist(X_w3_flat[:, sidx],   bins=50, alpha=0.6, label="Week 3 (drifted)",
            color="#F44336", density=True)

    psi_base_w3 = calculate_psi(X_base_flat[:, sidx], X_w3_flat[:, sidx])
    ax.set_title(f"Sensor {sidx}  PSI(baseline→W3)={psi_base_w3:.3f}",
                 fontweight="bold")
    ax.set_xlabel("Normalised value")
    ax.legend(fontsize=8)

plt.suptitle("Distribution Comparison: Baseline vs Production Weeks",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../logs/drift_reports/distribution_comparison.png", dpi=120, bbox_inches="tight")
plt.show()

## 6. Review Alert Logs

In [ ]:
report_dir = "../logs/drift_reports"
report_files = sorted([f for f in os.listdir(report_dir) if f.endswith(".json")])

print(f"Drift reports saved ({len(report_files)} files):")
for fname in report_files:
    fpath = os.path.join(report_dir, fname)
    with open(fpath) as f:
        rep = json.load(f)
    retrain_flag = "🔄 RETRAIN" if rep.get("retrain") else "✅ STABLE"
    print(f"  {fname:<50}  PSI={rep.get('psi_critical', '?'):2}  "
          f"KS={rep.get('ks_drifted', '?'):2}  {retrain_flag}")

print(f"\n🏁 Monitoring notebook complete.")
print("\n✅ Project pipeline end-to-end:")
print("  01_eda.ipynb           → Dataset generation & exploration")
print("  02_preprocessing.ipynb → Normalisation & feature engineering")
print("  03_training.ipynb      → Model training & evaluation")
print("  04_kafka_test.ipynb    → Online pipeline validation")
print("  05_monitoring.ipynb    → Drift detection & MLOps")